# Data Management & Data Cleaning – `manutenzioni` (Tutte le sedi)

**Obiettivo:** Eseguire l'analisi esplorativa (*Data Profiling*), la pulizia e la validazione degli interventi di manutenzione di tutte le sedi (**ALB, BAR, CAS, STR, TOR**) utilizzando esclusivamente `pandas` e `numpy`.

## 1. Import delle librerie e caricamento dati

In [1]:
import os
import numpy as np
import pandas as pd

SEDI = ["ALB", "BAR", "CAS", "STR", "TOR"]
BASE_DATI = "../buby_rent_dati_sporchi"
BASE_FILES = "../../files"
OUT_BASE = "../../dati_puliti"

flotta = pd.read_csv(f"{BASE_FILES}/flotta_con_tariffe.csv")
nol_clean = pd.read_csv(f"{OUT_BASE}/noleggi_tutti_pulito.csv")

df_raw = pd.concat(
    [pd.read_csv(f"{BASE_DATI}/{s}/strutturati/manutenzioni_{s}.csv").assign(_sede_origine=s) for s in SEDI],
    ignore_index=True
)

print(f"Dataset complessivo caricato: {df_raw.shape[0]} righe, {df_raw.shape[1]} colonne")
df_raw.head()

Dataset complessivo caricato: 881 righe, 14 colonne


,id_intervento,id_veicolo,data_ingresso,data_uscita,giorni_fermo,tipo_intervento,descrizione_breve,officina,km_veicolo,costo_eur,addebitato_cliente,id_noleggio_origine,stato_intervento,_sede_origine
0,INT-ALB-0001,ALB-05,03/11/2025,07/12/2025,34.0,CARROZZERIA,deformazione importante,Carrozzeria Gallo,32370,"6615,00 €",S,NOL-ALB-00001,NaN,ALB
1,INT-ALB-0002,alb-07,06/11/25,07/11/25,1.0,cambio gomme invernali,cambio stagionale,Autofficina Langhe,33.461,78.00,NO,NaN,chiuso,ALB
2,INT-ALB-0003,GJ812NP,2025-11-07,2025-11-13,6.0,Gomme,sostituzione pneumatico,Carrozzeria Gallo,110281,"146,00 €",NaN,NOL-ALB-00007,chiuso,ALB
3,INT-ALB-0084,EB-ALB-002,23/05/2026,24/05/2026,1.0,tagliando,tagliando programmato,Ciclofficina Alba,3394,"44,00",no,NaN,chiuso ok,ALB
4,INT-ALB-0005,ALB-08,09/11/2025,10/11/2025,1.0,cambio gomme invernali,cambio stagionale,Carrozzeria Gallo,51.628,"81,00",NO,NaN,CHIUSO,ALB


## 2. Data Exploration & Profiling iniziale

In [2]:
print("Valori mancanti per colonna:\n", df_raw.isna().sum()[df_raw.isna().sum() > 0])
print("\nDuplicati su id_intervento:", df_raw['id_intervento'].duplicated().sum())
print("\nDistribuzione tipo_intervento grezzo:\n", df_raw['tipo_intervento'].value_counts())

Valori mancanti per colonna:
 data_uscita             10
giorni_fermo            53
descrizione_breve       43
officina                38
km_veicolo              26
costo_eur               30
addebitato_cliente     164
id_noleggio_origine    356
stato_intervento       154
dtype: int64

Duplicati su id_intervento: 20

Distribuzione tipo_intervento grezzo:
 tipo_intervento
carrozzeria               119
tagliando                 115
bici                       68
meccanica                  38
carrozz.                   33
cambio gomme invernali     31
pulizia profonda           31
cambio gomme estive        30
assistenza                 27
Carrozzeria                26
CARROZZERIA                25
elettrico/batteria         24
manutenzione ordinaria     23
tagl.                      22
Pulizia Profonda           20
TAGLIANDO                  19
gomme                      19
Tagliando                  18
Bici                       17
vetri                      16
cerchi/gomme              

## 3. Pulizia chiavi (`id_intervento`, `id_veicolo`) e incrocio con anagrafiche

In [3]:
df = df_raw.copy()
df['id_intervento'] = df['id_intervento'].astype(str).str.strip().str.upper()

v_clean = df['id_veicolo'].astype('string').str.strip().str.upper().str.replace(r'\s+', '-', regex=True)
targa_map = flotta.dropna(subset=['targa']).set_index('targa')['id_veicolo']
valid_ids = set(flotta['id_veicolo'])

from_targa = v_clean.str.replace(r'[^A-Z0-9]', '', regex=True).map(targa_map)
eb_parts = v_clean.str.extract(r'^EB-?([A-Z]{3})-?(\d+)$')
auto_parts = v_clean.str.extract(r'^([A-Z]{3})-?(\d+)$')

norm_id = np.where(
    eb_parts[0].notna(),
    'EB-' + eb_parts[0].fillna('') + '-' + eb_parts[1].fillna('').str.zfill(3),
    np.where(
        auto_parts[0].notna(),
        auto_parts[0].fillna('') + '-' + auto_parts[1].fillna('').str.zfill(2),
        pd.NA
    )
)

df['id_veicolo'] = pd.Series(norm_id, index=df.index).fillna(from_targa)
df.loc[~df['id_veicolo'].isin(valid_ids), 'id_veicolo'] = pd.NA

# Recupero id_veicolo mancante tramite id_noleggio_origine (usando i noleggi già puliti)
df['id_noleggio_origine'] = df['id_noleggio_origine'].astype('string').str.strip().str.upper()
nol_map = nol_clean.dropna(subset=['id_noleggio', 'id_veicolo']).set_index('id_noleggio')['id_veicolo']
df['id_veicolo'] = df['id_veicolo'].fillna(df['id_noleggio_origine'].map(nol_map))

print("ID veicolo ancora mancanti dopo incrocio con flotta e noleggi:", df['id_veicolo'].isna().sum())

ID veicolo ancora mancanti dopo incrocio con flotta e noleggi: 3


## 4. Parsing date (`data_ingresso`, `data_uscita`) e ricalcolo `giorni_fermo`

In [4]:
mesi_it_num = {
    ' gen ': '/01/', ' feb ': '/02/', ' mar ': '/03/', ' apr ': '/04/',
    ' mag ': '/05/', ' giu ': '/06/', ' lug ': '/07/', ' ago ': '/08/',
    ' set ': '/09/', ' ott ': '/10/', ' nov ': '/11/', ' dic ': '/12/'
}

raw_in, raw_out = df['data_ingresso'].copy(), df['data_uscita'].copy()

def parse_dt(s, dayfirst=True):
    'Parsing robusto: le date ISO vengono lette esplicitamente perche '
    'format=mixed + dayfirst=True inverte giorno e mese su 2025-11-04 '
    '(comportamento diverso tra pandas 2 e pandas 3).'
    s = s.astype('string').str.strip().str.lower().replace({'-': pd.NA, 'n.d.': pd.NA})
    for k, v in mesi_it_num.items():
        s = s.str.replace(k, v, regex=False)
    iso = s.str.match(r'^\d{4}-\d{2}-\d{2}', na=False)
    out = pd.Series(pd.NaT, index=s.index, dtype='datetime64[ns]')
    out[iso] = pd.to_datetime(s[iso], format='ISO8601', errors='coerce')
    out[~iso] = pd.to_datetime(s[~iso], format='mixed', dayfirst=dayfirst, errors='coerce')
    return out

for col in ['data_ingresso', 'data_uscita']:
    df[col] = parse_dt(df[col])

# Risoluzione formato americano MM/DD/YYYY quando data_uscita < data_ingresso
mask_inv = df['data_ingresso'].notna() & df['data_uscita'].notna() & (df['data_uscita'] < df['data_ingresso'])
df.loc[mask_inv, 'data_ingresso'] = parse_dt(raw_in.loc[mask_inv], dayfirst=False)
df.loc[mask_inv, 'data_uscita'] = parse_dt(raw_out.loc[mask_inv], dayfirst=False)

mask_date = df['data_ingresso'].notna() & df['data_uscita'].notna()
df.loc[mask_date, 'giorni_fermo'] = (
    (df.loc[mask_date, 'data_uscita'] - df.loc[mask_date, 'data_ingresso']).dt.days.clip(lower=1).astype(float)
)

## 5. Standardizzazione categoriche, chilometri e costi (`costo_eur`)

In [5]:
mappa_tipo_int = {
    'carrozzeria': 'carrozzeria', 'carrozz.': 'carrozzeria',
    'tagliando': 'tagliando', 'tagl.': 'tagliando', 'manutenzione ordinaria': 'tagliando',
    'bici': 'bici', 'ciclofficina': 'bici',
    'meccanica': 'meccanica', 'meccanico': 'meccanica', 'mecc.': 'meccanica',
    'cambio gomme invernali': 'cambio gomme invernali', 'cambio gomme inv.': 'cambio gomme invernali', 'gomme invernali': 'cambio gomme invernali',
    'cambio gomme estive': 'cambio gomme estive', 'cambio gomme est.': 'cambio gomme estive', 'gomme estive': 'cambio gomme estive',
    'gomme': 'gomme', 'cerchi/gomme': 'gomme', 'gomme/cerchi': 'gomme', 'cerchi': 'gomme', 'pneumatici': 'gomme', 'cambio gomme': 'gomme',
    'pulizia profonda': 'pulizia profonda', 'pulizia straordinaria': 'pulizia profonda',
    'assistenza': 'assistenza', 'soccorso': 'assistenza',
    'elettrico/batteria': 'elettrico/batteria', 'elettrico': 'elettrico/batteria', 'batteria': 'elettrico/batteria',
    'vetri': 'vetri', 'cristalli': 'vetri', 'parabrezza': 'vetri'
}

df['tipo_intervento'] = df['tipo_intervento'].str.strip().str.lower().replace(mappa_tipo_int)
df['descrizione_breve'] = (
    df['descrizione_breve']
    .astype('string')
    .str.replace('&nbsp;', ' ', regex=False)
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.lower()
)
df['officina'] = (
    df['officina']
    .astype('string')
    .str.strip()
    .replace({'-': pd.NA, 'n.d.': pd.NA})
    .str.replace(r'\s+', ' ', regex=True)
    .str.title()
)
df['addebitato_cliente'] = (
    df['addebitato_cliente']
    .astype('string')
    .str.strip()
    .str.lower()
    .replace({'sì': 'si', 's': 'si', 'n': 'no'})
)
df['stato_intervento'] = (
    df['stato_intervento']
    .astype('string')
    .str.strip()
    .str.lower()
    .replace({'chiuso ok': 'chiuso'})
    .fillna(pd.Series(np.where(df['data_uscita'].notna(), 'chiuso', 'in corso'), index=df.index))
)

# Pulizia km_veicolo e costo_eur
s_km = df['km_veicolo'].astype('string').str.lower().str.replace('km', '', regex=False).str.replace(' ', '', regex=False).str.replace(r'(?<=\d)\.(?=\d{3}$)', '', regex=True)
df['km_veicolo'] = pd.to_numeric(s_km, errors='coerce')

s_eur = (
    df['costo_eur']
    .astype('string')
    .str.replace('€', '', regex=False)
    .str.replace(' ', '', regex=False)
    .str.replace(r'\.(?=\d{3},)', '', regex=True)
    .str.replace(',', '.', regex=False)
)
df['costo_eur'] = pd.to_numeric(s_eur, errors='coerce')

# Correzione outlier costo x10 rispetto alla mediana per tipo_intervento
mediana_tipo = df.groupby('tipo_intervento')['costo_eur'].transform('median')
mask_x10 = (df['costo_eur'] > mediana_tipo * 6) & (df['costo_eur'] > 1000)
df.loc[mask_x10, 'costo_eur'] = (df.loc[mask_x10, 'costo_eur'] / 10).round(2)

## 6. Deduplicazione, verifica qualità ed esportazione

In [6]:
df['_completezza'] = df.notna().sum(axis=1)
df_clean = (
    df.sort_values(['id_intervento', '_completezza'], ascending=[True, False])
    .drop_duplicates(subset=['id_intervento'], keep='first')
    .drop(columns=['_completezza'])
    .reset_index(drop=True)
)

log_gt = pd.concat([pd.read_csv(f"{BASE_DATI}/_ground_truth/log_difetti_strutturati_{s}.csv") for s in SEDI], ignore_index=True)
difetti_int = log_gt[log_gt['id_riga'].str.startswith('INT', na=False)]

print(f"Righe iniziali: {len(df_raw)} -> Righe pulite deduplicate: {len(df_clean)}")
print(f"Difetti iniettati nel log per le manutenzioni (conteggio, NON verifica di risoluzione): {len(difetti_int)}")
assert df_clean['id_intervento'].is_unique
assert (df_clean['data_uscita'] < df_clean['data_ingresso']).sum() == 0
assert (df_clean['giorni_fermo'].dropna() < 0).sum() == 0

cols_export = [c for c in df_clean.columns if c != '_sede_origine']
for s in SEDI:
    out_dir = f"{OUT_BASE}/{s}/strutturati"
    os.makedirs(out_dir, exist_ok=True)
    df_clean[df_clean['_sede_origine'] == s][cols_export].to_csv(f"{out_dir}/manutenzioni_{s}_pulito.csv", index=False)

df_clean[cols_export].to_csv(f"{OUT_BASE}/manutenzioni_tutti_pulito.csv", index=False)
print("Salvataggio completato in:", OUT_BASE)
df_clean[cols_export].head()

Righe iniziali: 881 -> Righe pulite deduplicate: 861
Difetti iniettati nel log per le manutenzioni (conteggio, NON verifica di risoluzione): 877
Salvataggio completato in: ../../dati_puliti


,id_intervento,id_veicolo,data_ingresso,data_uscita,giorni_fermo,tipo_intervento,descrizione_breve,officina,km_veicolo,costo_eur,addebitato_cliente,id_noleggio_origine,stato_intervento
0,INT-ALB-0001,ALB-05,2025-11-03,2025-12-07,34.0,carrozzeria,deformazione importante,Carrozzeria Gallo,32370,661.5,si,NOL-ALB-00001,chiuso
1,INT-ALB-0002,ALB-07,2025-11-06,2025-11-07,1.0,cambio gomme invernali,cambio stagionale,Autofficina Langhe,33461,78.0,no,<NA>,chiuso
2,INT-ALB-0003,ALB-04,2025-11-07,2025-11-13,6.0,gomme,sostituzione pneumatico,Carrozzeria Gallo,110281,146.0,<NA>,NOL-ALB-00007,chiuso
3,INT-ALB-0004,EB-ALB-003,2025-11-08,2025-11-15,7.0,bici,cambio posteriore rotto,<NA>,787,200.0,si,NOL-ALB-00039,chiuso
4,INT-ALB-0005,ALB-08,2025-11-09,2025-11-10,1.0,cambio gomme invernali,cambio stagionale,Carrozzeria Gallo,51628,81.0,no,<NA>,chiuso
